In [ ]:
# Ячейка 1: Импорты и загрузка
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
import lab_utils

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

feature_sets = lab_utils.load_feature_sets()
best_models = lab_utils.load_lab03_results()
calib_audit = pd.read_csv('outputs/calibration_audit.csv')

In [ ]:
# Ячейка 2: Подбор порога
datasets = {
    'cardiovascular_risk': (cardio.drop('target',axis=1), cardio['target']),
    'credit_risk': (credit.drop('default',axis=1), credit['default'])
}

threshold_results = []
policy_results = []

for ds_name, (X, y) in datasets.items():
    config = [m for m in best_models if m['dataset'] == ds_name][0]
    features = feature_sets[ds_name][config['feature_set']]
    
    X_temp, X_test, y_temp, y_test = train_test_split(X[features], y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    
    # Обучаем с лучшей калибровкой
    best_calib = calib_audit[
        (calib_audit['dataset'] == ds_name) & 
        (calib_audit['variant'] != 'uncalibrated')
    ].loc[calib_audit['brier'].idxmin()]
    
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_s, y_train)
    y_prob_val = model.predict_proba(X_val_s)[:, 1]
    
    # Перебор порогов
    for threshold in np.arange(0.1, 1.0, 0.05):
        threshold = round(threshold, 2)
        y_pred = (y_prob_val >= threshold).astype(int)
        
        threshold_results.append({
            'dataset': ds_name,
            'model': 'LogisticRegression',
            'variant': best_calib['variant'],
            'threshold': threshold,
            'precision': precision_score(y_val, y_pred, zero_division=0),
            'recall': recall_score(y_val, y_pred, zero_division=0),
            'f1': f1_score(y_val, y_pred, zero_division=0),
            'fp_rate': ((y_pred == 1) & (y_val == 0)).sum() / (y_val == 0).sum(),
            'fn_rate': ((y_pred == 0) & (y_val == 1)).sum() / (y_val == 1).sum(),
            'expected_cost': lab_utils.expected_cost(y_val.values, y_pred)
        })
    
    # Выбор лучшего порога: min cost при recall >= 0.60
    threshold_df = pd.DataFrame([r for r in threshold_results if r['dataset'] == ds_name])
    valid = threshold_df[threshold_df['recall'] >= 0.60]
    
    if len(valid) > 0:
        best_threshold = valid.loc[valid['expected_cost'].idxmin()]
    else:
        best_threshold = threshold_df.loc[threshold_df['expected_cost'].idxmin()]
    
    # Проверка на test
    y_prob_test = model.predict_proba(X_test_s)[:, 1]
    y_pred_test = (y_prob_test >= best_threshold['threshold']).astype(int)
    
    policy_results.append({
        'dataset': ds_name,
        'model': 'LogisticRegression',
        'variant': best_calib['variant'],
        'policy_name': 'min_cost_recall_60',
        'threshold': best_threshold['threshold'],
        'accuracy': (y_pred_test == y_test).mean(),
        'f1': f1_score(y_test, y_pred_test, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob_test),
        'pr_auc': 0,
        'expected_cost': lab_utils.expected_cost(y_test.values, y_pred_test),
        'cost_per_100': lab_utils.expected_cost(y_test.values, y_pred_test) * 100
    })
    
    # Сегментный анализ
    X_test_df = X_test.copy()
    X_test_df['age_segment'] = pd.cut(
        X_test_df['age'], bins=[0,45,60,100], labels=['young','middle','senior']
    )
    y_pred_series = pd.Series(y_pred_test, index=X_test.index)
    y_test_series = pd.Series(y_test.values, index=X_test.index)
    
    for segment in ['young','middle','senior']:
        mask = X_test_df['age_segment'] == segment
        if mask.sum() > 0:
            seg_pred = y_pred_series[mask]
            seg_true = y_test_series[mask]
            policy_results.append({
                'dataset': ds_name,
                'segment_feature': 'age',
                'segment': segment,
                'n': mask.sum(),
                'fp_rate': ((seg_pred == 1) & (seg_true == 0)).sum() / max((seg_true == 0).sum(), 1),
                'fn_rate': ((seg_pred == 0) & (seg_true == 1)).sum() / max((seg_true == 1).sum(), 1),
                'expected_cost_per_100': lab_utils.expected_cost(seg_true.values, seg_pred.values) * 100
            })

# Сохранение
pd.DataFrame(threshold_results).to_csv('outputs/threshold_policy_grid.csv', index=False)
pd.DataFrame([r for r in policy_results if 'policy_name' in r]).to_csv('outputs/policy_test_report.csv', index=False)
pd.DataFrame([r for r in policy_results if 'segment' in r]).to_csv('outputs/segment_policy_audit.csv', index=False)

print("Лучшие пороги:")
for r in [r for r in policy_results if 'policy_name' in r]:
    print(f"{r['dataset']}: threshold={r['threshold']}, cost={r['expected_cost']:.3f}")